# City-Hour Analysis: Hourly Air Quality (India, 2015–2020)

# Air Quality Index Analysis and Prediction

## Objectives
1. Data Cleaning: Handle missing values and format timestamps.
2. Exploratory Data Analysis: Identify daily and monthly fluctuations.
3. Machine Learning: Predict numerical AQI values and classify AQI severity categories.
4. Model Comparison: Benchmark multiple algorithms to identify the top performer.

Why hourly data? The daily file cannot show within-day behaviour (rush-hour peaks, night-time build-up). Hourly data can.

In [1]:
import pandas as pd, numpy as np, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt, seaborn as sns, os, warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, accuracy_score,
                             precision_recall_fscore_support, confusion_matrix, classification_report)
warnings.filterwarnings('ignore'); sns.set_theme(style='whitegrid'); RS=42

RAW='../data/raw/city_hour.csv'
if not os.path.exists(RAW): RAW='city_hour.csv'          # fallback if run next to the file
FIG='../outputs/figures/city_hour/'; RES='../outputs/results/'; PROC='../data/processed/'
for p in (FIG,RES,PROC): os.makedirs(p,exist_ok=True)
def save(name): plt.tight_layout(); plt.savefig(FIG+name,dpi=130); plt.show()

df=pd.read_csv(RAW); print(df.shape); df.head()

(707875, 16)


,City,Datetime,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
0,Ahmedabad,2015-01-01 01:00:00,NaN,NaN,1.00,40.01,36.37,NaN,1.00,122.07,NaN,0.0,0.0,0.0,NaN,NaN
1,Ahmedabad,2015-01-01 02:00:00,NaN,NaN,0.02,27.75,19.73,NaN,0.02,85.90,NaN,0.0,0.0,0.0,NaN,NaN
2,Ahmedabad,2015-01-01 03:00:00,NaN,NaN,0.08,19.32,11.08,NaN,0.08,52.83,NaN,0.0,0.0,0.0,NaN,NaN
3,Ahmedabad,2015-01-01 04:00:00,NaN,NaN,0.30,16.45,9.20,NaN,0.30,39.53,153.58,0.0,0.0,0.0,NaN,NaN
4,Ahmedabad,2015-01-01 05:00:00,NaN,NaN,0.12,14.90,7.85,NaN,0.12,32.63,NaN,0.0,0.0,0.0,NaN,NaN


1. Data cleaning


In [2]:
POLL=['PM2.5','PM10','NO','NO2','NOx','NH3','CO','SO2','O3','Benzene','Toluene','Xylene']
n0=len(df)
df['Datetime']=pd.to_datetime(df['Datetime'])                       # text -> real datetime
df['City']=df['City'].str.strip()                                   # standardise city names
df['AQI_Bucket']=df['AQI_Bucket'].str.strip()
dups=df.duplicated(['City','Datetime']).sum(); df=df.drop_duplicates(['City','Datetime'])
df=df.sort_values(['City','Datetime']).reset_index(drop=True)       # sort City then time
empty=df[POLL].isna().all(axis=1).sum(); df=df[~df[POLL].isna().all(axis=1)]   # rows with zero information
print(f'rows before: {n0:,} | duplicates removed: {dups} | fully-empty rows removed: {empty:,} | rows now: {len(df):,}')
print('Cities:',df.City.nunique(),'| Range:',df.Datetime.min(),'->',df.Datetime.max())

miss=(df[POLL+['AQI']].isna().mean()*100).round(1).sort_values(ascending=False)
print('\n% missing per column:'); print(miss)
miss.to_csv(RES+'city_hour_missing_pct.csv')

rows before: 707,875 | duplicates removed: 0 | fully-empty rows removed: 57,611 | rows now: 650,264
Cities: 26 | Range: 2015-01-01 01:00:00 -> 2020-07-01 00:00:00

% missing per column:
Xylene     61.2
PM10       36.8
NH3        33.1
Toluene    25.1
Benzene    16.3
PM2.5      13.5
AQI        12.1
SO2        11.2
O3         11.0
NOx        10.1
NO2         9.2
NO          9.1
CO          4.4
dtype: float64


In [3]:
# Decision: drop columns that are too empty to be reliable (>50% missing overall)
drop_cols=[c for c in POLL if df[c].isna().mean()>0.5]
print('Dropping:',drop_cols,'(too empty - imputing would mostly be invention)')
POLL=[c for c in POLL if c not in drop_cols]; df=df.drop(columns=drop_cols)

# Per-city coverage check (guide 4.2): which city-pollutant pairs are mostly empty?
cov=df.groupby('City')[POLL+['AQI']].apply(lambda g:(g.isna().mean()*100).round(0))
plt.figure(figsize=(10,8)); sns.heatmap(cov,annot=True,fmt='.0f',cmap='Reds',cbar_kws={'label':'% missing'})
plt.title('Missing data (%) per city and column'); save('01_missing_by_city.png')

Dropping: ['Xylene'] (too empty - imputing would mostly be invention)


2. Exploratory Data Analysis (EDA)
Outliers are kept here on purpose - pollution spikes are real patterns.

In [4]:
df['Hour']=df.Datetime.dt.hour; df['Month']=df.Datetime.dt.month; df['Year']=df.Datetime.dt.year
df['Weekday']=df.Datetime.dt.dayofweek

# 2.1 City-wise average AQI
city_aqi=df.groupby('City').AQI.mean().sort_values(ascending=False)
plt.figure(figsize=(9,7)); sns.barplot(x=city_aqi.values,y=city_aqi.index,color='tab:red')
plt.xlabel('Mean hourly AQI'); plt.title('Average AQI by city (2015-2020)'); save('02_city_avg_aqi.png')
print(city_aqi.round(1).head(5))

City
Ahmedabad    455.0
Delhi        260.2
Patna        236.7
Gurugram     223.6
Lucknow      218.4
Name: AQI, dtype: float64


In [5]:
# 2.2 Hour-of-day pattern (the main benefit of hourly data)
fig,ax=plt.subplots(1,3,figsize=(15,4))
for a,c in zip(ax,['PM2.5','NO2','CO']):
    df.groupby('Hour')[c].mean().plot(ax=a,marker='o'); a.set_title(f'Average {c} by hour of day'); a.set_xlabel('Hour')
save('03_hourly_pollutant_profile.png')
print(df.groupby('Hour')[['PM2.5','NO2','CO']].mean().round(1).T)

Hour     0     1     2     3     4     5     6     7     8     9   ...    14  \
PM2.5  79.8  76.6  74.9  70.1  67.4  65.6  65.8  67.9  72.1  74.8  ...  56.7   
NO2    33.3  30.2  28.0  26.1  25.0  24.8  25.7  27.1  28.5  28.5  ...  22.5   
CO      3.5   2.9   2.2   2.0   1.7   1.7   1.7   1.9   2.2   2.0  ...   1.4   

Hour     15    16    17    18    19    20    21    22    23  
PM2.5  54.2  52.2  52.1  54.4  61.0  70.5  77.4  80.6  81.1  
NO2    22.4  23.1  25.0  29.8  36.6  40.6  40.4  38.3  35.9  
CO      1.4   1.4   1.4   1.5   1.9   3.0   4.0   4.2   3.9  

[3 rows x 24 columns]


In [6]:
# 2.3 Hour x Month heatmap of PM2.5 - when is the air worst?
pv=df.pivot_table(index='Hour',columns='Month',values='PM2.5',aggfunc='mean')
plt.figure(figsize=(10,7)); sns.heatmap(pv,cmap='YlOrRd',cbar_kws={'label':'Mean PM2.5'}); plt.title('PM2.5 by hour and month'); save('04_hour_month_heatmap.png')

# 2.4 Delhi vs others: daily rhythm
sel=[c for c in ['Delhi','Mumbai','Chennai','Kolkata','Ahmedabad'] if c in df.City.unique()]
plt.figure(figsize=(9,5))
for c in sel: df[df.City==c].groupby('Hour')['PM2.5'].mean().plot(label=c)
plt.legend(); plt.ylabel('Mean PM2.5'); plt.title('Daily PM2.5 rhythm by city'); save('05_city_hourly_pm25.png')

In [7]:
# 2.5 Seasonality + weekday effect + yearly trend
fig,ax=plt.subplots(1,3,figsize=(16,4))
sns.boxplot(data=df.dropna(subset=['AQI']),x='Month',y='AQI',ax=ax[0],showfliers=False); ax[0].set_title('AQI by month')
df.groupby('Weekday').AQI.mean().plot(kind='bar',ax=ax[1],color='tab:blue'); ax[1].set_title('Mean AQI by weekday (0=Mon)')
df.groupby('Year').AQI.mean().plot(marker='o',ax=ax[2]); ax[2].set_title('Mean AQI by year (2020 = COVID lockdown)')
save('06_month_weekday_year.png')
print(df.groupby('Month').AQI.mean().round(0).to_dict()); print(df.groupby('Year').AQI.mean().round(0).to_dict())

# 2.6 Class balance + correlation heatmap
plt.figure(figsize=(8,4)); order=['Good','Satisfactory','Moderate','Poor','Very Poor','Severe']
sns.countplot(data=df.dropna(subset=['AQI_Bucket']),x='AQI_Bucket',order=order,color='tab:orange'); plt.title('AQI_Bucket distribution'); save('07_bucket_distribution.png')
plt.figure(figsize=(9,7)); corr=df[POLL+['AQI']].corr(); sns.heatmap(corr,annot=True,fmt='.2f',cmap='coolwarm',center=0); plt.title('Correlation matrix'); save('08_correlation.png')
corr['AQI'].sort_values(ascending=False).round(2)

{1: 231.0, 2: 203.0, 3: 163.0, 4: 143.0, 5: 135.0, 6: 120.0, 7: 112.0, 8: 114.0, 9: 116.0, 10: 189.0, 11: 241.0, 12: 226.0}
{2015: 210.0, 2016: 197.0, 2017: 180.0, 2018: 184.0, 2019: 157.0, 2020: 114.0}


AQI        1.00
PM10       0.67
CO         0.58
PM2.5      0.50
NO2        0.43
NOx        0.39
SO2        0.38
NO         0.34
Toluene    0.26
NH3        0.23
O3         0.09
Benzene    0.05
Name: AQI, dtype: float64

**Reading the EDA:** PM2.5/PM10 and CO track AQI most closely; pollutants peak in the morning and late evening (traffic + shallow night-time boundary layer); winter months are worst; 2020 is cleaner due to the COVID lockdown. (See printed numbers above and the saved figures.)

## 3. Feature engineering
Time features from the datetime (hourly data lets us add hour-based features the daily file cannot have).

In [8]:
season={12:'Winter',1:'Winter',2:'Winter',3:'Summer',4:'Summer',5:'Summer',6:'Monsoon',7:'Monsoon',8:'Monsoon',9:'PostMonsoon',10:'PostMonsoon',11:'PostMonsoon'}
df['Season']=df.Month.map(season)
df['is_winter']=df.Month.isin([11,12,1,2]).astype(int)
df['is_weekend']=(df.Weekday>=5).astype(int)
df['is_rush_hour']=df.Hour.isin([7,8,9,17,18,19,20]).astype(int)
df['is_night']=df.Hour.isin([22,23,0,1,2,3,4,5]).astype(int)
# cyclic encoding so that hour 23 and hour 0 are 'close'
df['hour_sin']=np.sin(2*np.pi*df.Hour/24); df['hour_cos']=np.cos(2*np.pi*df.Hour/24)
df=pd.get_dummies(df,columns=['Season'],prefix='season')
df.to_csv(PROC+'city_hour_clean.csv.gz',index=False,compression='gzip')
print('Saved cleaned+featured data:',df.shape)

Saved cleaned+featured data: (650264, 29)


## 4. Modelling set-up
* **Targets:** `AQI` (regression) and `AQI_Bucket` (classification). Rows where the target is missing are **dropped, never imputed**.
* **Predictors** (pollutants) with missing values are filled with the **median of that city + month**, then the city median. Medians are learned from the *training period only* to avoid leakage.
* **Split:** time-based - train on 2015-2019, test on 2020 (time-series data is never shuffled). A random-split score is reported for reference in Section 7.
* **Note:** CPCB hourly AQI is built from rolling 24-hour pollutant averages, so AQI is smooth hour to hour - but the 2020 test year is unusual (COVID lockdown), so time-based scores are lower than a random split would suggest (see Section 7).

In [9]:
m=df.dropna(subset=['AQI','AQI_Bucket']).copy()
print('Rows with a target:',len(m))
train_mask=m.Year<=2019
med_cm=m[train_mask].groupby(['City','Month'])[POLL].median()
med_c =m[train_mask].groupby('City')[POLL].median()
def impute(d):
    d=d.copy()
    key=pd.MultiIndex.from_frame(d[['City','Month']])
    for c in POLL:
        fill=pd.Series(med_cm[c].reindex(key).values,index=d.index)
        d[c]=d[c].fillna(fill).fillna(d.City.map(med_c[c])).fillna(m[train_mask][c].median())
    return d
m=impute(m)
city_dum=pd.get_dummies(m.City,prefix='city').astype(int)
TIME=['Hour','Month','Weekday','is_winter','is_weekend','is_rush_hour','is_night','hour_sin','hour_cos']+[c for c in m if c.startswith('season_')]
X=pd.concat([m[POLL+TIME].astype(float),city_dum],axis=1)
tr,te=(m.Year<=2019).values,(m.Year==2020).values
print('Train:',tr.sum(),'| Test (2020):',te.sum())
rng=np.random.RandomState(RS); tr_idx=np.where(tr)[0]; sub=rng.choice(tr_idx,size=min(100000,len(tr_idx)),replace=False)
Xtr,Xte=X.iloc[sub],X[te]; sc=StandardScaler().fit(Xtr)

Rows with a target: 571476


Train: 469335 | Test (2020): 102141


## 5. Regression - predict the AQI value

In [10]:
y=m.AQI; ytr,yte=y.iloc[sub],y[te]
models={'Linear Regression (baseline)':(LinearRegression(),True),
        'Decision Tree':(DecisionTreeRegressor(max_depth=12,min_samples_leaf=20,random_state=RS),False),
        'Random Forest':(RandomForestRegressor(n_estimators=60,max_depth=18,min_samples_leaf=5,n_jobs=-1,random_state=RS),False)}
rows=[];preds={}
for n,(mod,scaled) in list(models.items()):
    A,B=(sc.transform(Xtr),sc.transform(Xte)) if scaled else (Xtr,Xte)
    mod.fit(A,ytr); p=mod.predict(B); preds[n]=p; models[n]=(mod,scaled)
    rows.append([n,mean_absolute_error(yte,p),np.sqrt(mean_squared_error(yte,p)),r2_score(yte,p)])
reg=pd.DataFrame(rows,columns=['Model','MAE','RMSE','R2']).round(3); reg.to_csv(RES+'city_hour_regression_results.csv',index=False); reg

,Model,MAE,RMSE,R2
0,Linear Regression (baseline),40.565,62.904,0.517
1,Decision Tree,34.547,62.835,0.518
2,Random Forest,33.234,58.787,0.578


In [11]:
best=reg.sort_values('RMSE').iloc[0].Model; p=preds[best]
fig,ax=plt.subplots(1,2,figsize=(12,5))
idx=np.random.RandomState(0).choice(len(yte),5000,replace=False)
ax[0].scatter(yte.values[idx],p[idx],s=4,alpha=.4); ax[0].plot([0,800],[0,800],'r--'); ax[0].set_xlabel('Actual AQI'); ax[0].set_ylabel('Predicted'); ax[0].set_title(f'{best}: actual vs predicted (2020 test)')
ax[1].hist(yte.values-p,bins=60); ax[1].set_title('Residuals (actual - predicted)')
save('09_regression_actual_vs_pred.png')

rf=models['Random Forest'][0]; imp=pd.Series(rf.feature_importances_,index=X.columns).sort_values(ascending=False)
imp.to_csv(RES+'city_hour_feature_importance_reg.csv'); plt.figure(figsize=(8,6)); imp.head(15)[::-1].plot(kind='barh',color='tab:green'); plt.title('Random Forest feature importance (regression)'); save('10_feature_importance_reg.png'); imp.head(10).round(3)

PM2.5             0.353
CO                0.222
city_Ahmedabad    0.138
NO                0.046
NO2               0.039
Hour              0.038
PM10              0.029
NOx               0.019
O3                0.017
Toluene           0.016
dtype: float64

## 6. Classification - predict the AQI category

In [12]:
yc=m.AQI_Bucket; yctr,ycte=yc.iloc[sub],yc[te]
cmodels={'Logistic Regression (baseline)':(LogisticRegression(max_iter=300),True),
         'Decision Tree':(DecisionTreeClassifier(max_depth=12,min_samples_leaf=20,random_state=RS),False),
         'Random Forest':(RandomForestClassifier(n_estimators=60,max_depth=18,min_samples_leaf=5,n_jobs=-1,random_state=RS),False)}
rows=[];cpred={}
for n,(mod,scaled) in list(cmodels.items()):
    A,B=(sc.transform(Xtr),sc.transform(Xte)) if scaled else (Xtr,Xte)
    mod.fit(A,yctr); p=mod.predict(B); cpred[n]=p
    pr,rc,f1,_=precision_recall_fscore_support(ycte,p,average='macro',zero_division=0)
    rows.append([n,accuracy_score(ycte,p),pr,rc,f1])
clf=pd.DataFrame(rows,columns=['Model','Accuracy','Precision(macro)','Recall(macro)','F1(macro)']).round(3)
clf.to_csv(RES+'city_hour_classification_results.csv',index=False); clf

,Model,Accuracy,Precision(macro),Recall(macro),F1(macro)
0,Logistic Regression (baseline),0.643,0.588,0.538,0.530
1,Decision Tree,0.660,0.574,0.540,0.551
2,Random Forest,0.667,0.634,0.540,0.548


In [13]:
bestc=clf.sort_values('F1(macro)').iloc[-1].Model; p=cpred[bestc]
labels=['Good','Satisfactory','Moderate','Poor','Very Poor','Severe']; labels=[l for l in labels if l in set(ycte)|set(p)]
cm=confusion_matrix(ycte,p,labels=labels)
plt.figure(figsize=(7,6)); sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=labels,yticklabels=labels); plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title(f'Confusion matrix - {bestc} (2020 test)'); save('11_confusion_matrix.png')
rep=pd.DataFrame(classification_report(ycte,p,labels=labels,output_dict=True,zero_division=0)).T.round(3)
rep.to_csv(RES+'city_hour_classification_report_best.csv'); print(bestc); rep

Decision Tree


,precision,recall,f1-score,support
Good,0.709,0.446,0.547,13054.00
Satisfactory,0.691,0.774,0.730,44022.00
Moderate,0.667,0.687,0.677,34206.00
Poor,0.385,0.288,0.329,6626.00
Very Poor,0.462,0.483,0.473,3485.00
Severe,0.531,0.566,0.548,748.00
accuracy,0.660,0.660,0.660,0.66
macro avg,0.574,0.540,0.551,102141.00
weighted avg,0.656,0.660,0.653,102141.00


## 7. Robustness check - random split vs time-based split

In [14]:
Xa,Xb,ya,yb=train_test_split(X.iloc[sub],y.iloc[sub],test_size=0.2,random_state=RS)
rf2=RandomForestRegressor(n_estimators=60,max_depth=18,min_samples_leaf=5,n_jobs=-1,random_state=RS).fit(Xa,ya)
rs=pd.DataFrame({'Split':['Random 80/20','Time-based (test = 2020)'],
                 'RF R2':[r2_score(yb,rf2.predict(Xb)),reg.set_index('Model').loc['Random Forest','R2']]}).round(3)
rs.to_csv(RES+'city_hour_split_comparison.csv',index=False); rs

,Split,RF R2
0,Random 80/20,0.787
1,Time-based (test = 2020),0.578


## 8. Conclusions
* **Regression (test = 2020):** Random Forest is best (MAE ~33, RMSE ~59, R² ~0.58) vs the linear baseline (MAE ~41, R² ~0.52); Decision Tree is in between.
* **Classification:** Random Forest has the best accuracy (~0.67); Decision Tree has a slightly higher macro-F1 (~0.55). "Poor" is the hardest class (neighbouring-category confusion); "Severe" is rare.
* **Important features:** PM2.5 and CO dominate; `city_Ahmedabad` also ranks high, meaning city identity matters (Ahmedabad's AQI levels differ systematically).
* **Random vs time-based split:** R² drops from ~0.79 (random) to ~0.58 (time-based) - random splits overstate performance for time-series data, so the time-based numbers are the honest ones.
* **Limitations:** ~12% of hourly AQI is missing; Xylene (61% missing) dropped; the test year 2020 is atypical; features are not lagged, so this is not true forecasting.